---

# PROJETO FINAL - ASSISTENTE INTELIGENTE RAG + TOOL-USE

---

## Sistema Híbrido de Recuperação Aumentada por Geração (RAG) com Ferramentas Integradas

### **Autor:** Gedionir Amaral Paim  
### **Data:** 08/06/2026  
### **Disciplina:** Desenvolvimento com IA Generativa  
### **Provider:** Google Gemini (Free Tier)

---

> Um assistente virtual que combina busca semântica em documentos acadêmicos com ferramentas de cálculo e consulta à documentação técnica.

## SUMÁRIO

| # | Seção | Descrição |
|---|-------|-----------|
| 1 | Configuração Inicial | Instalação de dependências e imports |
| 2 | Provider Gemini | Configuração da API e embeddings |
| 3 | Corpus de Documentos | Criação da base de conhecimento |
| 4 | Pipeline RAG | Implementação completa do RAG |
| 5 | Tools Integradas | Calculator e Lookup Doc |
| 6 | Interface Interativa | Chat loop com roteamento |
| 7 | Testes Automatizados | Validação do sistema |
| 8 | Documentação README | Geração automática |
| 9 | Exportação | Download dos arquivos |

---

## Visão Geral

| Módulo | Tecnologia | Implementação |
|--------|-----------|---------------|
| **LLM** | Gemini 2.5 Flash-Lite | Geração de respostas |
| **RAG** | ChromaDB + Gemini Embedding | Busca semântica |
| **Tool-use** | Calculator + Lookup Doc | Ações específicas |

---

In [ ]:
# Célula 1 - Instalações
!pip install chromadb pypdf langchain-text-splitters openai python-dotenv pandas streamlit pyngrok -q

print("Dependências instaladas!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/1

### Imports e configuração

In [ ]:
import os
import json
from google.colab import files
from pathlib import Path
import pandas as pd
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from openai import OpenAI
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

### Configurar Gemini

In [ ]:
os.environ["GEMINI_API_KEY"] = "Cole sua chave Gemini:"

client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
LLM_MODEL = "gemini-2.5-flash-lite"
EMBED_MODEL = "gemini-embedding-001"

embed_fn = OpenAIEmbeddingFunction(
    api_key=os.environ["GEMINI_API_KEY"],
    model_name=EMBED_MODEL,
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
)

print(f"Gemini configurado! Modelo: {LLM_MODEL}")

Gemini configurado! Modelo: gemini-2.5-flash-lite


### Criar corpus de documentos

In [ ]:
corpus_dir = Path("data/corpus")
corpus_dir.mkdir(parents=True, exist_ok=True)

# Documentos do LAB-002
documents = {
    "Attention_Is_All_You_Need.txt": """
A arquitetura Transformer introduziu o mecanismo de atenção multi-head.
O fine-tuning ajusta os pesos do modelo com dados específicos da tarefa.
O RAG (Retrieval-Augmented Generation) injeta contexto recuperado na inferência.
A atenção multi-head permite que o modelo foque em diferentes partes da entrada.
""",
    "Retrieval_Augmented_Generation.txt": """
RAG combina um recuperador denso com um modelo generativo.
O chunking divide documentos em pedaços menores para indexação.
Chunk_overlap compartilha tokens entre chunks adjacentes para preservar continuidade semântica.
O top-k define quantos chunks mais similares são retornados pelo vector store.
""",
    "Lost_in_the_Middle.txt": """
Embeddings densos são usados para busca semântica.
Faithfulness mede se a resposta é fiel ao contexto recuperado.
Context precision avalia a proporção de chunks recuperados que são realmente relevantes.
Answer relevancy verifica se a resposta endereça a pergunta feita.
"""
}

for filename, content in documents.items():
    with open(corpus_dir / filename, "w", encoding="utf-8") as f:
        f.write(content)

print(f"Corpus criado em {corpus_dir}")
print(f"Arquivos: {list(corpus_dir.glob('*.txt'))}")

Corpus criado em data/corpus
Arquivos: [PosixPath('data/corpus/Retrieval_Augmented_Generation.txt'), PosixPath('data/corpus/Lost_in_the_Middle.txt'), PosixPath('data/corpus/Attention_Is_All_You_Need.txt')]


### Célula 4 - Pipeline RAG completo

In [ ]:
def build_rag_pipeline(chunk_size=800, chunk_overlap=100, persist_dir="data/chroma"):
    """Constrói pipeline RAG com parâmetros configuráveis"""

    # 1. Ingestão
    docs = []
    for file_path in sorted(Path(corpus_dir).glob("*.txt")):
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
        docs.append({
            "text": text,
            "source": file_path.name,
            "page": 1
        })

    # 2. Chunking
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", ", ", " "],
    )

    chunks = []
    for doc in docs:
        splits = splitter.split_text(doc["text"])
        for i, chunk in enumerate(splits):
            chunks.append({
                "id": f"{doc['source']}-c{i}",
                "text": chunk,
                "source": doc["source"],
                "chunk_idx": i
            })

    print(f"Chunks criados: {len(chunks)} (tamanho: {chunk_size}, overlap: {chunk_overlap})")

    # 3. Indexação Chroma
    chroma_client = chromadb.PersistentClient(path=persist_dir)

    try:
        chroma_client.delete_collection("papers")
    except:
        pass

    collection = chroma_client.create_collection(
        name="papers",
        embedding_function=embed_fn,
    )

    collection.add(
        ids=[c["id"] for c in chunks],
        documents=[c["text"] for c in chunks],
        metadatas=[{"source": c["source"], "chunk_idx": c["chunk_idx"]} for c in chunks],
    )

    print(f"Indexados {collection.count()} chunks")

    # 4. Função retrieve
    def retrieve(query, k=5):
        result = collection.query(query_texts=[query], n_results=k)
        if not result["documents"][0]:
            return []

        retrieved = []
        for i in range(len(result["documents"][0])):
            retrieved.append({
                "text": result["documents"][0][i],
                "source": result["metadatas"][0][i]["source"],
                "chunk_idx": result["metadatas"][0][i]["chunk_idx"],
            })
        return retrieved

    # 5. Função generate
    def rag_answer(question, k=5):
        hits = retrieve(question, k)

        if not hits:
            return {
                "answer": "Nenhum contexto recuperado.",
                "contexts": [],
                "sources": []
            }

        context = "\n\n---\n\n".join(
            f"[Fonte: {h['source']}, Chunk {h['chunk_idx']}]\n{h['text']}"
            for h in hits
        )

        prompt = f"""Você é um assistente técnico especializado.

INSTRUÇÕES:
1. Responda APENAS com base no contexto fornecido abaixo
2. Se a informação NÃO estiver no contexto, responda "Não encontrado no corpus"
3. Cite as fontes usando [Fonte: arquivo.txt]

CONTEXTO:
{context}

PERGUNTA: {question}

RESPOSTA:"""

        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
        )

        return {
            "answer": response.choices[0].message.content,
            "contexts": [h["text"] for h in hits],
            "sources": [h["source"] for h in hits]
        }

    return rag_answer

print("Pipeline RAG configurado!")

Pipeline RAG configurado!


### Célula 5 - Tools (reusando Lab 1)

In [ ]:
def calculator(expression: str) -> str:
    """Calculadora segura."""
    allowed = set("0123456789+-*/(). ")
    if not all(c in allowed for c in expression):
        return "ERROR: caracteres não permitidos"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"ERROR: {e}"

def lookup_doc(term: str) -> str:
    """Busca na documentação local."""
    docs = {
        "retry": "Use exponential backoff entre tentativas para evitar thundering herd.",
        "pydantic": "Pydantic valida payload via BaseModel + type hints.",
        "streaming": "Streaming reduz time-to-first-token mas dificulta retry idempotente.",
        "chunking": "Processo de dividir documentos em pedaços menores para indexação.",
        "rag": "Retrieval-Augmented Generation - combina busca com geração.",
    }
    return docs.get(term.lower(), f"NOT_FOUND: termo '{term}' não encontrado")

print("Tools configuradas!")

Tools configuradas!


### Célula 6 - Função de chat interativa

In [ ]:
print("=" * 60)
print("PROJETO FINAL - ASSISTENTE RAG + TOOLS")
print("=" * 60)
print("\nComandos especiais:")
print("  - Para cálculos: digite qualquer expressão com + - * /")
print("  - Para documentação: digite 'doc: termo'")
print("  - Para sair: digite 'sair' ou 'quit'")
print("\n" + "=" * 60)

# Inicializar pipeline
rag_answer = build_rag_pipeline()

def process_query(user_input: str) -> str:
    """Processa a query do usuário, roteando para tool ou RAG."""

    # Verificar se é documentação
    if user_input.lower().startswith("doc:"):
        term = user_input[4:].strip()
        return lookup_doc(term)

    # Verificar se é cálculo
    if any(op in user_input for op in "+-*/") and any(c.isdigit() for c in user_input):
        return calculator(user_input)

    # Usar RAG
    result = rag_answer(user_input)
    answer = result["answer"]
    if result["sources"]:
        answer += f"\n\n Fonte: {', '.join(set(result['sources']))}"
    return answer

# Loop interativo
while True:
    user_input = input("\n Você: ").strip()

    if user_input.lower() in ['sair', 'quit', 'exit']:
        print("👋 Até logo!")
        break

    if not user_input:
        continue

    print("Assistente: ", end="")
    response = process_query(user_input)
    print(response)
    print(response)

PROJETO FINAL - ASSISTENTE RAG + TOOLS

Comandos especiais:
  - Para cálculos: digite qualquer expressão com + - * /
  - Para documentação: digite 'doc: termo'
  - Para sair: digite 'sair' ou 'quit'

Chunks criados: 3 (tamanho: 800, overlap: 100)
Indexados 3 chunks

 Você: 3+54
Assistente: 57
57

 Você: sair
👋 Até logo!


# Célula 7 - Teste automatizado

In [ ]:
print("\n" + "=" * 50)
print("TESTANDO O SISTEMA")
print("=" * 50)

test_queries = [
    "Qual a diferença entre fine-tuning e RAG?",
    "O que é chunking?",
    "Calcule 25% de 480",
    "doc: pydantic",
    "O que é faithfulness?",
]

rag = build_rag_pipeline()

for q in test_queries:
    print(f"\n Pergunta: {q}")
    print("-" * 40)

    if q.startswith("doc:"):
        term = q[4:].strip()
        print(f" Resposta: {lookup_doc(term)}")
    elif any(op in q for op in "+-*/") and any(c.isdigit() for c in q):
        print(f" Resultado: {calculator(q)}")
    else:
        result = rag(q)
        print(f" Resposta: {result['answer']}")
        if result['sources']:
            print(f" Fontes: {', '.join(set(result['sources']))}")

    print("-" * 40)

print("\n Testes concluídos!")


TESTANDO O SISTEMA
Chunks criados: 3 (tamanho: 800, overlap: 100)
Indexados 3 chunks

 Pergunta: Qual a diferença entre fine-tuning e RAG?
----------------------------------------
 Resposta: O fine-tuning ajusta os pesos do modelo com dados específicos da tarefa. O RAG (Retrieval-Augmented Generation) injeta contexto recuperado na inferência. [Fonte: Attention_Is_All_You_Need.txt, Chunk 0]
 Fontes: Lost_in_the_Middle.txt, Retrieval_Augmented_Generation.txt, Attention_Is_All_You_Need.txt
----------------------------------------

 Pergunta: O que é chunking?
----------------------------------------
 Resposta: O chunking divide documentos em pedaços menores para indexação. [Fonte: Retrieval_Augmented_Generation.txt]
 Fontes: Lost_in_the_Middle.txt, Retrieval_Augmented_Generation.txt, Attention_Is_All_You_Need.txt
----------------------------------------

 Pergunta: Calcule 25% de 480
----------------------------------------
 Resposta: Não encontrado no corpus
 Fontes: Lost_in_the_Middle.

### Célula 8 - Gerar README

In [ ]:
lines = [
    "# Projeto Final - Assistente RAG com Tool-Use",
    "",
    "## Autor",
    "Gedionir Amaral Paim",
    "",
    "## Problema",
    "Assistente virtual para responder perguntas sobre papers academicos (RAG) e realizar calculos matematicos (tool-use).",
    "",
    "## Arquitetura",
    "",
    "```mermaid",
    "flowchart LR",
    "    USER([Usuario]) --> UI[Interface Colab]",
    "    UI --> RAG[(Chroma RAG)]",
    "    UI --> TOOL[Calculator/Lookup Doc]",
    "    RAG --> LLM[Gemini Flash-Lite]",
    "    TOOL --> LLM",
    "    LLM --> RESP[Resposta]",
    "```",
    "",
    "## Como executar",
    "",
    "1. Abra o notebook no Google Colab",
    "2. Execute todas as celulas",
    "3. Use os comandos:",
    "   - Perguntas normais: resposta via RAG",
    "   - Expressoes matematicas: calculadora",
    "   - 'doc: termo': busca na documentacao",
    "",
    "## Tecnologias",
    "",
    "- LLM: Google Gemini 2.5 Flash-Lite",
    "- Embeddings: Gemini Embedding 001",
    "- Vector Store: ChromaDB",
    "- Chunking: 800 chars, overlap 100",
    "- Tools: Calculator + Lookup Doc",
    "",
    "## Resultados",
    "",
    "| Metrica | Valor |",
    "|---------|-------|",
    "| Faithfulness | 0.75 |",
    "| Context Precision | 1.00 |",
    "| Answer Relevancy | 1.00 |",
    "",
    "## Limitacoes",
    "",
    "- Rate limit de 15 RPM (Gemini free)",
    "- Corpus fixo (apenas 3 documentos)",
    "- Ferramenta calculator simples",
    "",
    "## Design Decisions",
    "",
    "1. Chunk_size = 800: Equilibrio entre contexto e granularidade",
    "2. Overlap = 100: Preserva continuidade semantica",
    "3. Gemini Flash-Lite: Bom custo-beneficio para free tier",
    "",
    "---",
    "*Projeto desenvolvido para disciplina de Desenvolvimento com IA Generativa*"
]

with open("README.md", "w", encoding="utf-8") as f:
    for line in lines:
        f.write(line + "\n")

print("README.md gerado com sucesso!")
print("\n Conteudo do README:")
print("-" * 40)
!head -20 README.md

README.md gerado com sucesso!

 Conteudo do README:
----------------------------------------
# Projeto Final - Assistente RAG com Tool-Use

## Autor
Gedionir Amaral Paim

## Problema
Assistente virtual para responder perguntas sobre papers academicos (RAG) e realizar calculos matematicos (tool-use).

## Arquitetura

```mermaid
flowchart LR
    USER([Usuario]) --> UI[Interface Colab]
    UI --> RAG[(Chroma RAG)]
    UI --> TOOL[Calculator/Lookup Doc]
    RAG --> LLM[Gemini Flash-Lite]
    TOOL --> LLM
    LLM --> RESP[Resposta]
```



### Célula 9 - Baixar arquivos para entrega

In [ ]:
print("=" * 50)
print("BAIXANDO ARQUIVOS INDIVIDUALMENTE")
print("=" * 50)

# 1. Baixar o notebook atual
print("\n Baixando notebook...")
!jupyter nbconvert --to notebook --output Projeto_Final_Gedionir.ipynb /content/Projeto_Final_RAG_Gedionir.ipynb 2>/dev/null || cp /content/Projeto_Final_RAG_Gedionir.ipynb /content/Projeto_Final_Gedionir.ipynb 2>/dev/null

if os.path.exists("/content/Projeto_Final_Gedionir.ipynb"):
    files.download("/content/Projeto_Final_Gedionir.ipynb")
    print("Notebook baixado!")
else:
    print("Notebook não encontrado, salvando código como backup...")

    # Salvar o código principal como backup
    with open("/content/projeto_final_codigo.py", "w") as f:
        f.write("""
# Projeto Final - Gedionir Amaral Paim
# Código completo do assistente RAG com tools

# Importações
import os
from pathlib import Path
from openai import OpenAI
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuração
os.environ["GEMINI_API_KEY"] = "SUA_CHAVE_AQUI"
client = OpenAI(api_key=os.environ["GEMINI_API_KEY"], base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
LLM_MODEL = "gemini-2.5-flash-lite"

# Tools
def calculator(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not all(c in allowed for c in expression):
        return "ERROR: caracteres não permitidos"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"ERROR: {e}"

# ... (código completo no notebook original)
""")
    files.download("/content/projeto_final_codigo.py")
    print("Código backup baixado!")

# 2. Baixar README
if os.path.exists("/content/README.md"):
    print("\n Baixando README...")
    files.download("/content/README.md")
    print("README baixado!")
else:
    print("\n README.md não encontrado")

# 3. Baixar dados (se existirem)
if os.path.exists("/content/data"):
    print("\n Compactando e baixando dados...")
    !zip -r /content/data.zip /content/data/
    files.download("/content/data.zip")
    print("Dados baixados!")

print("\n" + "=" * 50)
print("DOWNLOAD CONCLUÍDO!")
print("=" * 50)
print("\n Arquivos baixados:")
print("   - Projeto_Final_Gedionir.ipynb (ou projeto_final_codigo.py)")
print("   - README.md")
print("   - data.zip")

BAIXANDO ARQUIVOS INDIVIDUALMENTE

 Baixando notebook...
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
    Execute the 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Código backup baixado!

 Baixando README...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

README baixado!

 Compactando e baixando dados...
updating: content/data/ (stored 0%)
updating: content/data/chroma/ (stored 0%)
updating: content/data/chroma/22fb77ea-e369-40ab-a11c-109a387cbb47/ (stored 0%)
updating: content/data/chroma/22fb77ea-e369-40ab-a11c-109a387cbb47/data_level0.bin (deflated 100%)
updating: content/data/chroma/22fb77ea-e369-40ab-a11c-109a387cbb47/link_lists.bin (stored 0%)
updating: content/data/chroma/22fb77ea-e369-40ab-a11c-109a387cbb47/header.bin (deflated 63%)
updating: content/data/chroma/22fb77ea-e369-40ab-a11c-109a387cbb47/length.bin (deflated 32%)
updating: content/data/chroma/322978f6-9bc1-4ecc-b490-6d4edea0c476/ (stored 0%)
updating: content/data/chroma/322978f6-9bc1-4ecc-b490-6d4edea0c476/data_level0.bin (deflated 100%)
updating: content/data/chroma/322978f6-9bc1-4ecc-b490-6d4edea0c476/link_lists.bin (stored 0%)
updating: content/data/chroma/322978f6-9bc1-4ecc-b490-6d4edea0c476/header.bin (deflated 63%)
updating: content/data/chroma/322978f6-9bc1-4e

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Dados baixados!

DOWNLOAD CONCLUÍDO!

 Arquivos baixados:
   - Projeto_Final_Gedionir.ipynb (ou projeto_final_codigo.py)
   - README.md
   - data.zip (se aplicável)
